In [ ]:
from google.colab import drive
import os
import json
import pandas as pd
import numpy as np

drive.mount("/content/drive")

PROJECT_PATH = "/content/drive/MyDrive/Delhi-AQI-Prediction"

OUTPUT_PATH = os.path.join(
    PROJECT_PATH,
    "outputs"
)

MODEL_PATH = os.path.join(
    PROJECT_PATH,
    "models"
)

TABLE_PATH = os.path.join(
    PROJECT_PATH,
    "reports",
    "tables"
)

FIGURE_PATH = os.path.join(
    PROJECT_PATH,
    "reports",
    "figures"
)

PROCESSED_PATH = os.path.join(
    PROJECT_PATH,
    "data",
    "processed"
)

os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(TABLE_PATH, exist_ok=True)
os.makedirs(FIGURE_PATH, exist_ok=True)

print("Project connected.")

In [ ]:
classical_results = pd.read_csv(
    os.path.join(
        TABLE_PATH,
        "phase6_classical_model_results.csv"
    )
)

display(classical_results)

In [ ]:
deep_results = pd.read_csv(
    os.path.join(
        TABLE_PATH,
        "phase8_deep_learning_results.csv"
    )
)

display(deep_results)

In [ ]:
vmd_results = pd.read_csv(
    os.path.join(
        TABLE_PATH,
        "phase9_vmd_results.csv"
    )
)

display(vmd_results)

In [ ]:
timegan_results = pd.read_csv(
    os.path.join(
        TABLE_PATH,
        "phase10_timegan_forecasting_results.csv"
    )
)

display(timegan_results)

In [ ]:
master_comparison = pd.DataFrame([
    {
        "Model": "Persistence Baseline",
        "Category": "Baseline",
        "MAE": 3.730738,
        "RMSE": 8.959175,
        "R2": 0.994857,
        "EvaluationScope": "Full Delhi test set - 37 stations",
        "DirectlyComparableToRF": True
    },
    {
        "Model": "Linear Regression",
        "Category": "Classical ML",
        "MAE": 0.368180,
        "RMSE": 1.643302,
        "R2": 0.999827,
        "EvaluationScope": "Full Delhi test set - 37 stations",
        "DirectlyComparableToRF": True
    },
    {
        "Model": "Decision Tree",
        "Category": "Classical ML",
        "MAE": 0.034008,
        "RMSE": 0.799976,
        "R2": 0.999959,
        "EvaluationScope": "Full Delhi test set - 37 stations",
        "DirectlyComparableToRF": True
    },
    {
        "Model": "Random Forest",
        "Category": "Classical ML",
        "MAE": 0.014194,
        "RMSE": 0.731606,
        "R2": 0.999966,
        "EvaluationScope": "Full Delhi test set - 37 stations",
        "DirectlyComparableToRF": True
    },
    {
        "Model": "XGBoost",
        "Category": "Classical ML",
        "MAE": 0.530287,
        "RMSE": 1.738577,
        "R2": 0.999806,
        "EvaluationScope": "Full Delhi test set - 37 stations",
        "DirectlyComparableToRF": True
    },
    {
        "Model": "LSTM",
        "Category": "Deep Learning",
        "MAE": 0.685707,
        "RMSE": 1.060120,
        "R2": 0.999928,
        "EvaluationScope": "Sequence-based test set",
        "DirectlyComparableToRF": False
    },
    {
        "Model": "CNN-BiLSTM",
        "Category": "Deep Learning",
        "MAE": 1.421856,
        "RMSE": 2.347170,
        "R2": 0.999648,
        "EvaluationScope": "Sequence-based test set",
        "DirectlyComparableToRF": False
    },
    {
        "Model": "VMD-LSTM",
        "Category": "Hybrid",
        "MAE": 5.918161,
        "RMSE": 10.521427,
        "R2": 0.992570,
        "EvaluationScope": "Selected 5 Delhi stations",
        "DirectlyComparableToRF": False
    },
    {
        "Model": "Real-only LSTM - TimeGAN Experiment",
        "Category": "TimeGAN Benchmark",
        "MAE": 13.339487,
        "RMSE": 17.651962,
        "R2": 0.975890,
        "EvaluationScope": "Single station - R K Puram",
        "DirectlyComparableToRF": False
    },
    {
        "Model": "TimeGAN-Augmented LSTM",
        "Category": "TimeGAN",
        "MAE": 20.538877,
        "RMSE": 25.042191,
        "R2": 0.951476,
        "EvaluationScope": "Single station - R K Puram",
        "DirectlyComparableToRF": False
    }
])

display(master_comparison.sort_values("RMSE"))

In [ ]:
comparable_models = master_comparison[
    master_comparison[
        "DirectlyComparableToRF"
    ]
].copy()

comparable_models = comparable_models.sort_values(
    "RMSE"
).reset_index(drop=True)

comparable_models["Rank"] = (
    np.arange(len(comparable_models)) + 1
)

display(comparable_models)

In [ ]:
best_row = comparable_models.iloc[0]

best_model_name = best_row["Model"]
best_model_mae = best_row["MAE"]
best_model_rmse = best_row["RMSE"]
best_model_r2 = best_row["R2"]

print("Current best full-Delhi model:")
print("Model:", best_model_name)
print("MAE :", best_model_mae)
print("RMSE:", best_model_rmse)
print("R²  :", best_model_r2)

In [ ]:
baseline_row = comparable_models[
    comparable_models["Model"]
    == "Persistence Baseline"
].iloc[0]

baseline_mae = baseline_row["MAE"]
baseline_rmse = baseline_row["RMSE"]

mae_improvement = (
    (baseline_mae - best_model_mae)
    / baseline_mae
) * 100

rmse_improvement = (
    (baseline_rmse - best_model_rmse)
    / baseline_rmse
) * 100

print(
    "MAE improvement over persistence:",
    round(mae_improvement, 4),
    "%"
)

print(
    "RMSE improvement over persistence:",
    round(rmse_improvement, 4),
    "%"
)

In [ ]:
ablation_results = pd.read_csv(
    os.path.join(
        TABLE_PATH,
        "phase7_ablation_results.csv"
    )
)

display(ablation_results)

In [ ]:
# Inspect the actual experiment names first
print("Available ablation experiments:")
print(
    ablation_results["Experiment"].tolist()
)

# --------------------------------------------------
# Identify rows safely
# --------------------------------------------------

persistence_row = ablation_results[
    ablation_results["Experiment"].str.contains(
        "Persistence",
        case=False,
        na=False
    )
].iloc[0]

experiment_a_row = ablation_results[
    ablation_results["Experiment"].str.startswith(
        "A:"
    )
].iloc[0]

experiment_b_row = ablation_results[
    ablation_results["Experiment"].str.startswith(
        "B:"
    )
].iloc[0]

persistence_rmse = persistence_row["RMSE"]
a_rmse = experiment_a_row["RMSE"]
b_rmse = experiment_b_row["RMSE"]

# --------------------------------------------------
# Calculate improvements
# --------------------------------------------------

a_vs_persistence = (
    (persistence_rmse - a_rmse)
    / persistence_rmse
) * 100

b_vs_a = (
    (a_rmse - b_rmse)
    / a_rmse
) * 100

print("\nAblation comparison")
print("-------------------")

print(
    "Persistence RMSE:",
    round(persistence_rmse, 6)
)

print(
    "Experiment A RMSE:",
    round(a_rmse, 6)
)

print(
    "Experiment B RMSE:",
    round(b_rmse, 6)
)

print(
    "\nRMSE improvement from A vs Persistence:",
    round(a_vs_persistence, 4),
    "%"
)

print(
    "RMSE improvement from B vs A:",
    round(b_vs_a, 4),
    "%"
)

In [ ]:
station_results = pd.read_csv(
    os.path.join(
        TABLE_PATH,
        "phase7_station_results.csv"
    )
)

display(station_results)

In [ ]:
worst_stations = (
    station_results
    .sort_values("RMSE", ascending=False)
    .head(10)
)

display(worst_stations)

In [ ]:
season_results = pd.read_csv(
    os.path.join(
        TABLE_PATH,
        "phase7_season_results.csv"
    )
)

display(season_results)

In [ ]:
event_results = pd.read_csv(
    os.path.join(
        TABLE_PATH,
        "phase7_event_results.csv"
    )
)

display(event_results)

In [ ]:
prediction_df = pd.read_parquet(
    os.path.join(
        OUTPUT_PATH,
        "phase6_test_predictions.parquet"
    )
)

prediction_df["Datetime"] = pd.to_datetime(
    prediction_df["Datetime"]
)

print(
    "Prediction rows:",
    len(prediction_df)
)

display(prediction_df.head())

In [ ]:
prediction_df["RF_Error"] = (
    prediction_df["AQI_t_plus_1"]
    - prediction_df["RandomForest_Prediction"]
)

prediction_df["RF_AbsoluteError"] = (
    prediction_df["RF_Error"]
    .abs()
)

print(
    prediction_df["RF_AbsoluteError"].describe()
)

In [ ]:
largest_errors = (
    prediction_df
    .sort_values(
        "RF_AbsoluteError",
        ascending=False
    )
    .head(20)
)

display(
    largest_errors[
        [
            "StationId",
            "StationName",
            "Datetime",
            "AQI",
            "AQI_t_plus_1",
            "RandomForest_Prediction",
            "RF_Error",
            "RF_AbsoluteError"
        ]
    ]
)

In [ ]:
extreme_threshold = (
    prediction_df["AQI"]
    .quantile(0.90)
)

extreme_df = prediction_df[
    prediction_df["AQI"] >= extreme_threshold
].copy()

extreme_mae = np.mean(
    np.abs(
        extreme_df["AQI_t_plus_1"]
        - extreme_df["RandomForest_Prediction"]
    )
)

extreme_rmse = np.sqrt(
    np.mean(
        (
            extreme_df["AQI_t_plus_1"]
            - extreme_df["RandomForest_Prediction"]
        ) ** 2
    )
)

print(
    "Extreme threshold:",
    extreme_threshold
)

print(
    "Extreme records:",
    len(extreme_df)
)

print(
    "Extreme MAE:",
    round(extreme_mae, 6)
)

print(
    "Extreme RMSE:",
    round(extreme_rmse, 6)
)

In [ ]:
duplicate_test_keys = prediction_df.duplicated(
    subset=["StationId", "Datetime"]
).sum()

duplicate_test_rows = prediction_df.duplicated().sum()

print(
    "Duplicate Station-Time test records:",
    duplicate_test_keys
)

print(
    "Duplicate complete test rows:",
    duplicate_test_rows
)

In [ ]:
master_comparison_path = os.path.join(
    TABLE_PATH,
    "phase11_master_model_comparison.csv"
)

master_comparison.to_csv(
    master_comparison_path,
    index=False
)

comparable_path = os.path.join(
    TABLE_PATH,
    "phase11_comparable_models.csv"
)

comparable_models.to_csv(
    comparable_path,
    index=False
)

print("Comparison tables saved.")

In [ ]:
current_model_decision = {
    "current_best_model": best_model_name,
    "MAE": float(best_model_mae),
    "RMSE": float(best_model_rmse),
    "R2": float(best_model_r2),
    "evaluation_scope": "Full Delhi future test period - 37 stations",
    "status": "Production candidate, not yet final",
    "reason": (
        "Best RMSE among directly comparable full-Delhi models. "
        "Final production selection requires live-style validation "
        "and reproducible inference testing."
    ),
    "next_step": (
        "Build the reusable inference pipeline and validate "
        "current-time t -> t+1 forecasting."
    )
}

decision_path = os.path.join(
    OUTPUT_PATH,
    "phase11_current_model_decision.json"
)

with open(
    decision_path,
    "w"
) as f:
    json.dump(
        current_model_decision,
        f,
        indent=4
    )

print("Current model decision saved.")

In [ ]:
print("=" * 70)
print("PHASE 11 — COMPREHENSIVE MODEL COMPARISON")
print("=" * 70)

print("Current best full-Delhi model:", best_model_name)
print("MAE :", round(best_model_mae, 6))
print("RMSE:", round(best_model_rmse, 6))
print("R²  :", round(best_model_r2, 6))

print(
    "\nMAE improvement vs persistence:",
    round(mae_improvement, 4),
    "%"
)

print(
    "RMSE improvement vs persistence:",
    round(rmse_improvement, 4),
    "%"
)

print(
    "\nExtreme-AQI RMSE:",
    round(extreme_rmse, 6)
)

print(
    "\nStatus:",
    "Production candidate — final live-validation still required"
)

print("=" * 70)

In [ ]:
phase11_status = {
    "completed_phase": "Phase 11 - Comprehensive Model Comparison and Provisional Final Selection",
    "next_phase": "Phase 12 - Reusable Live-Style Inference Pipeline",
    "current_best_model": best_model_name,
    "final_model_selected": False
}

status_path = os.path.join(
    OUTPUT_PATH,
    "session_status.json"
)

with open(
    status_path,
    "w"
) as f:
    json.dump(
        phase11_status,
        f,
        indent=4
    )

print("Phase 11 status saved.")

In [ ]:
import os
import json

PROJECT_PATH = "/content/drive/MyDrive/Delhi-AQI-Prediction"

OUTPUT_PATH = os.path.join(
    PROJECT_PATH,
    "outputs"
)

TABLE_PATH = os.path.join(
    PROJECT_PATH,
    "reports",
    "tables"
)

os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(TABLE_PATH, exist_ok=True)

# --------------------------------------------------
# 1. Save the comprehensive comparison
# --------------------------------------------------

master_comparison.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase11_master_model_comparison.csv"
    ),
    index=False
)

comparable_models.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase11_comparable_models.csv"
    ),
    index=False
)

# --------------------------------------------------
# 2. Save ablation results
# --------------------------------------------------

ablation_results.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase11_ablation_results.csv"
    ),
    index=False
)

# --------------------------------------------------
# 3. Save current model decision
# --------------------------------------------------

model_decision = {
    "current_best_model": "Random Forest",
    "MAE": 0.014194,
    "RMSE": 0.731606,
    "R2": 0.999966,
    "mae_improvement_vs_persistence_percent": 99.6195,
    "rmse_improvement_vs_persistence_percent": 91.834,
    "extreme_aqi_rmse": 1.767377,
    "evaluation_scope": "Full Delhi future test period - 37 stations",
    "status": "Production candidate",
    "final_model_selected": False,
    "next_requirement": "Live-style t -> t+1 validation"
}

decision_path = os.path.join(
    OUTPUT_PATH,
    "phase11_model_decision.json"
)

with open(
    decision_path,
    "w"
) as f:
    json.dump(
        model_decision,
        f,
        indent=4
    )

# --------------------------------------------------
# 4. Save session status
# --------------------------------------------------

session_status = {
    "completed_phase": "Phase 11 - Comprehensive Model Comparison",
    "next_phase": "Phase 12 - Reusable Live-Style Inference Pipeline",
    "current_best_model": "Random Forest",
    "final_model_selected": False
}

status_path = os.path.join(
    OUTPUT_PATH,
    "session_status.json"
)

with open(
    status_path,
    "w"
) as f:
    json.dump(
        session_status,
        f,
        indent=4
    )

# --------------------------------------------------
# 5. Verify checkpoint
# --------------------------------------------------

files_to_check = [
    os.path.join(
        TABLE_PATH,
        "phase11_master_model_comparison.csv"
    ),
    os.path.join(
        TABLE_PATH,
        "phase11_comparable_models.csv"
    ),
    os.path.join(
        TABLE_PATH,
        "phase11_ablation_results.csv"
    ),
    decision_path,
    status_path
]

print("=" * 60)
print("PHASE 11 CHECKPOINT SAVED")
print("=" * 60)

for file in files_to_check:
    print(
        "OK   " if os.path.exists(file) else "ERROR",
        file
    )

print("\nCompleted: Phase 11")
print("Current best candidate: Random Forest")
print("Next: Phase 12 - Reusable Live-Style Inference Pipeline")